In [271]:
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [272]:
# 사이킷런을 이용해 당뇨병 데이터 다운로드
diabetes = fetch_openml(
    data_id=37,
    as_frame=True,
    parser="pandas"
)

# 특성 데이터 복사
df = diabetes.data.copy()

# 정답 추가 (0 = 음성, 1 = 양성)
df["target"] = diabetes.target.map({
    "tested_negative": 0,
    "tested_positive": 1
}).astype(int)

print(df.shape)
print(df.head(3))

(768, 9)
   preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1


## 1. 데이터탐색

In [273]:
# 전체 행 개수와 열 개수 확인
print(df.shape)

(768, 9)


In [274]:
# 상위 5행 확인
print(df.head())

   preg  plas  pres  skin  insu  mass   pedi  age  target
0     6   148    72    35     0  33.6  0.627   50       1
1     1    85    66    29     0  26.6  0.351   31       0
2     8   183    64     0     0  23.3  0.672   32       1
3     1    89    66    23    94  28.1  0.167   21       0
4     0   137    40    35   168  43.1  2.288   33       1


In [275]:
# 각 열의 데이터 타입 확인
print(df.dtypes)

preg        int64
plas        int64
pres        int64
skin        int64
insu        int64
mass      float64
pedi      float64
age         int64
target      int64
dtype: object


In [276]:
# describe()로 기술 통계량 확인
print(df.describe().T)

        count        mean         std     min       25%       50%        75%  \
preg    768.0    3.845052    3.369578   0.000   1.00000    3.0000    6.00000   
plas    768.0  120.894531   31.972618   0.000  99.00000  117.0000  140.25000   
pres    768.0   69.105469   19.355807   0.000  62.00000   72.0000   80.00000   
skin    768.0   20.536458   15.952218   0.000   0.00000   23.0000   32.00000   
insu    768.0   79.799479  115.244002   0.000   0.00000   30.5000  127.25000   
mass    768.0   31.992578    7.884160   0.000  27.30000   32.0000   36.60000   
pedi    768.0    0.471876    0.331329   0.078   0.24375    0.3725    0.62625   
age     768.0   33.240885   11.760232  21.000  24.00000   29.0000   41.00000   
target  768.0    0.348958    0.476951   0.000   0.00000    0.0000    1.00000   

           max  
preg     17.00  
plas    199.00  
pres    122.00  
skin     99.00  
insu    846.00  
mass     67.10  
pedi      2.42  
age      81.00  
target    1.00  


In [277]:
# target 값별 데이터 개수 확인
print(df['target'].value_counts())

target
0    500
1    268
Name: count, dtype: int64


In [278]:
# 양성(1)과 음성(0)의 비율을 백분율로 출력
ratio = df['target'].value_counts(normalize=True) * 100
print(ratio.round(4))

target
0    65.1042
1    34.8958
Name: proportion, dtype: float64


분석질문
1. 전체 데이터에서 양성(1)의 비율은 몇 %인가요?

양성비율은 34.8958%로 전체에서 3분의 1 정도가 양성입니다.

2. 입력 특성 중 숫자 범위가 가증 큰 특성과 가장 작은 특성은 무엇인가요?

범위가 가장 큰 특성은 "insu"입니다. insu의 최솟값은 0이고 최대값은 846으로 범위가 846입니다.
범위가 가장 작은 특성은 "pedi"입니다. pedi의 최솟값은 0.078이고 최대값은 2.42로 범위가 2.342입니다.

3. 특성마다 숫자 범위가 다른 것이 KNN에 어떤 영향을 줄 수 있을까요?

KNN은 가까운 이웃을 찾는 알고리즘이고 값의 스케일이 큰 특성이 더 큰 영향을 끼치게 됩니다. 그래서 insu같이 스케일이 큰 특성이 이웃을 정하는데 거의 전부를 좌우하고, pedi같은 작은 특성은 사실상 무시되는 문제가 있습니다.
중요한 특성이 숫자범위가 작다는 이유로 무시될 수 있기 때문에, 이를 막기 위해선 스케일링이 필요합니다.

## 2. 결측치 탐색

In [279]:
# 판다스 기본 결측치(NaN) 개수를 열별로 확인
print(df.isna().sum())


preg      0
plas      0
pres      0
skin      0
insu      0
mass      0
pedi      0
age       0
target    0
dtype: int64


In [280]:
# 안내한 5개 열의 0 개수 각각 출력
print("plas : ", (df["plas"] == 0).sum())
print("pres : ", (df["pres"] == 0).sum())
print("skin : ", (df["skin"] == 0).sum())
print("insu : ", (df["insu"] == 0).sum())
print("mass : ", (df["mass"] == 0).sum())

plas :  5
pres :  35
skin :  227
insu :  374
mass :  11


In [281]:
# 해당 5개열에서 0을 NaN으로 변경
cols = ["plas", "pres", "skin", "insu", "mass"]
df[cols] = df[cols].replace(0, float("nan"))


In [282]:
# 0이 NaN으로 바뀌었는지 isna().sum()으로 재확인
print(df.isna().sum())

preg        0
plas        5
pres       35
skin      227
insu      374
mass       11
pedi        0
age         0
target      0
dtype: int64


In [283]:
# 결측치가 하나라도 포함된 행의 개수
print(df.isna().any(axis=1).sum())


376


분석 질문
1. 처음 isna()를 실행했을 때 결측치가 발견되지 않은 이유는 무엇인가요?

데이터에 빈칸이 아니라 0이라는 숫자로 저장되어 있었기 때문입니다. 즉 결측치가 없는 게 아니라 0으로 위장되어 있었습니다.

2. 모든 열의 0을 Nan으로 변경하면 안 되는 이유는 무엇인가요?

혈당, 혈압, 피부 두께 같은 값은 0이 될 수 없는 값이기 때문에 0을 결측치로 보는 것이 올바른 판단입니다. 하지만 "preg"의 0은 '임신횟수 0'이라는 정상값이고, target의 0 또한 음성이라는 옳은 값입니다. 따라서 preg와 target과 같은 칼럼에서 진짜 정보의 데이터가 훼손되는 경우가 있을 수 있기 때문에 0을 일괄적으로 NaN으로 변경해서는 안 됩니다. 

## 3. 특성과 정답 분리 및 데이터 분할

In [284]:
# target을 제외한 열들을 X에 저장
X = df.drop(columns=["target"])

# target을 y에 저장
y = df["target"]

# X와 y의 shape 출력
print("X.shape : ", X.shape)
print("y.shape : ", y.shape)


X.shape :  (768, 8)
y.shape :  (768,)


In [285]:
# 학습 80%, 테스트 20%으로 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 네 객체의 shape 출력
print("X_train.shape : ", X_train.shape)
print("X_test.shape : ", X_test.shape)
print("y_train.shape : ", y_train.shape)
print("y_test.shape : ", y_test.shape)

X_train.shape :  (614, 8)
X_test.shape :  (154, 8)
y_train.shape :  (614,)
y_test.shape :  (154,)


In [286]:
# 학습 데이터와 테스트 데이터의 양성/음성 개수 출력
print("학습 데이터")
print(y_train.value_counts())
print("테스트 데이터")
print(y_test.value_counts())

학습 데이터
target
0    400
1    214
Name: count, dtype: int64
테스트 데이터
target
0    100
1     54
Name: count, dtype: int64


분석질문
1. 학습 데이터와 테스트 데이터를 나누는 이유는 무엇인가요?

모델이 학습 중에 한 번도 보지 못한 데이터에서도 잘 맞히는지(일반화 성능)를 확인하기 위해서입니다. 학습에 쓴 데이터로만 평가하면 외운 결과가 그대로 나와 성능이 실제보다 좋게 보이므로, 학습에 쓰지 않은 데이터를 따로 떼어 두어야 과적합 여부를 판단할 수 있습니다.

2. stratify=y를 사용하는 이유는 무엇인가요?

학습데이터와 테스트데이터에 양성,음성의 비율을 원본과 같게 만들기 위해서 입니다. 이런 설정을 하지 않으면, 원래는 소수인 양성이 한쪽에 몰려 오히려 다수가 되거나 하는 경우가 생길수 있어 평가 지표가 운에 좌우되고 왜곡된 결과를 만들 수 있기 때문입니다.


3. random_state를 지정하면 어떤 장점이 있나요?

분할 결과가 매번 동일하게 재현됩니다. 같은 코드를 다시 실행해도 같은 X_train, X_test가 나오므로, 이후 k값이나 스케일링 방법을 바꿔 가며 성능을 비교할 때 (랜덤값을 고정해놨으니)'데이터가 달라서 생긴 차이'가 아닌 '설정이 달라지면서 생긴 차이'로 명확하게 비교하며 볼 수 있는 장점이 있습니다.

## 4. 결측치 처리

In [287]:
# 학습 데이터에서 각 특성의 중앙값 계산
X_train_median = X_train.median()
X_train_median


preg      3.0000
plas    117.0000
pres     72.0000
skin     29.0000
insu    125.0000
mass     32.4000
pedi      0.3825
age      29.0000
dtype: float64

In [288]:
# 학습 데이터 결측치를 중앙값으로 채우기
X_train_filled = X_train.fillna(X_train_median)

In [289]:
# 테스트 데이터 결측치도 학습 데이터 중앙값으로 채우기
X_test_filled = X_test.fillna(X_train_median)


In [290]:
# 처리 후 학습 데이터와 테스트 데이터 결측치 개수
print("학습 데이터 결측치 개수")
print(X_train_filled.isna().sum())
print("테스트 데이터 결측치 개수")
print(X_test_filled.isna().sum())


학습 데이터 결측치 개수
preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64
테스트 데이터 결측치 개수
preg    0
plas    0
pres    0
skin    0
insu    0
mass    0
pedi    0
age     0
dtype: int64


In [291]:
# 처리 전후 행 개수 비교
print("처리 전:", X_train.shape, X_test.shape)
print("처리 후:", X_train_filled.shape, X_test_filled.shape)

처리 전: (614, 8) (154, 8)
처리 후: (614, 8) (154, 8)


분석 질문
1. 결측치를 삭제하지 않고 중앙값으로 채우면 어떤 장점이 있나요?

데이터의 양을 그대로 유지할 수 있습니다. 이 데이터는 결측치가 있는 행이 전체의 절반 가까이라서 삭제하면 학습할 데이터가 크게 줄어듭니다. 그리고 중앙값으로 결측치를 채우는 것은 평균보다 이상치의 영향을 덜 받아서, "insu"처럼 값이 한쪽으로 치우친 특성을 다루는 데에도 적합합니다.


2. 테스트 데이터의 중앙값을 따로 계산해서 사용하면 안 되는 이유는 무엇인가요?

테스트 데이터는 학습과정에서 "아직 보지 못한 새 데이터"로 쓰이는 평가용 데이터인데, 여기서 중앙값을 계산해서 쓰면 테스트 데이터의 분포 정보가 전처리에 섞여 들어가는 데이터 누수가 생기게 됩니다. 그러면 실제 환경보다 유리한 조건에서 평가하게 되어 성능이 낙관적으로 부풀려집니다.

## 5. 첫번째 KNN 학습

In [292]:
# kNN 객체 생성
model = KNeighborsClassifier(n_neighbors=5)

In [293]:
# fit()으로 학습
model.fit(X_train_filled, y_train)

# 학습 데이터에 대한 정확도
base_train_acc = model.score(X_train_filled, y_train)
print("학습정확도 : ", round(base_train_acc,4))

학습정확도 :  0.8013


In [294]:
# 테스트 데이터에 대한 정확도
base_test_acc = model.score(X_test_filled, y_test)
print("테스트 정확도 : ", round(base_test_acc,4))

테스트 정확도 :  0.6753


In [295]:
# 테스트 데이터 예측
pred = model.predict(X_test_filled)

In [296]:
# 예측 결과와 실제 정답을 앞 10개씩 출력
print("예측 결과 : ", pred[:10])
print("실제 정답 : ", y_test.values[:10])

예측 결과 :  [1 0 0 0 0 0 0 1 0 1]
실제 정답 :  [0 0 0 1 0 0 1 1 0 0]


분석 질문
1. KNN에서 k=5는 어떤 의미인가요?

새 데이터를 분류할 때 학습 데이터 중 거리가 가장 가까운 5개를 찾아, 그 5개 중 더 많은 쪽으로 예측한다는 뜻입니다.

2. 학습 정확도와 테스트 정확도는 각각 무엇을 의미하나요?

학습 정확도는 모델이 이미 본 데이터를 얼마나 잘 맞히는지(암기 정도), 테스트 정확도는 한 번도 보지 못한 데이터를 얼마나 잘 맞히는지(일반화 성능)을 뜻합니다.

3. 학습 정확도와 테스트 정확도에 차이가 발생하는 이유는 무엇인가요?

학습 정확도를 잴 때는 각 샘플의 가장 가까운 이웃 5개 안에 자기 자신이 반드시 포함되어 거리 0으로 들어가므로 점수가 낙관적으로 나옵니다. 반면 테스트 데이터는 학습에 쓰이지 않은 새로운 샘플이라 점수가 낮아져서 보통 학습 정확도가 더 높게 나옵니다. 두 점수의 차이가 클수록 과대적합 가능성도 커집니다.

## 6. KNN 성능 평가

In [297]:
# 혼동행렬
cm = confusion_matrix(y_test, pred)
print(cm)

[[77 23]
 [27 27]]


In [298]:
# 정확도
print("정확도:", round(accuracy_score(y_test, pred),4))

정확도: 0.6753


In [299]:
# 정밀도
print("정밀도:", round(precision_score(y_test, pred),4))

정밀도: 0.54


In [300]:
# 재현율
base_recall = recall_score(y_test, pred)
print("재현율:", round(base_recall,4))

재현율: 0.5


In [301]:
# F1 점수
print("F1:", round(f1_score(y_test, pred),4))

F1: 0.5192


In [302]:
# classification_report
print(classification_report(y_test, pred, digits=4))

              precision    recall  f1-score   support

           0     0.7404    0.7700    0.7549       100
           1     0.5400    0.5000    0.5192        54

    accuracy                         0.6753       154
   macro avg     0.6402    0.6350    0.6371       154
weighted avg     0.6701    0.6753    0.6723       154



분석 질문
1. 실제 양성인데 음성으로 잘못 예측한 데이터(FN)는 몇 개입니까?

27개입니다.

2. 실제 음성인데 양성으로 잘못 예측한 데이터(FP)는 몇 개입니까?

23개입니다.

3. 정확도는 높지만 재현율은 낮은 상황이 발생할 수 있나요?

있습니다. 정확도는 전체에서 맞힌 비율이고 재현율은 실제 양성 중에서 맞게 찾아낸 비율이라 서로 값이 다를 수 있습니다. 이 모델이 바로 그 경우로, 정확도는 0.675이지만 재현율은 0.5입니다. 양성 54명 중 27명을 놓쳤는데도 음성 100명을 77명으로 높게 맞혀서, 정확도는 상대적으로 높게 나옵니다. 양성이 35% 뿐인 데이터라 다수 클래스(음성)만 잘 맞혀도 정확도는 올라가서 이런 차이가 발생합니다.

4. 당뇨병 양성을 놓치는 것을 줄이려면 어떤 평가 지표를 중요하게 봐야 할까요?

재현율을 우선 봐야합니다. 당뇨병 양성을 놓치면 치료 기회를 잃게 되므로, 놓친 양성(FN)을 줄이는 재현율이 핵심입니다. 정밀도와의 균형이 필요하면 F1 점수를 함께 봅니다.

## 7. k값에 따른 성능 비교

In [303]:
k_list = [1, 3, 5, 7, 10, 15, 20, 30]
k_results = []

# k값 외에는 동일한 조건 유지
for k in k_list:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_filled, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train_filled))
    test_acc = accuracy_score(y_test, model.predict(X_test_filled))

    k_results.append((k, train_acc, test_acc))
    print(f"{k} / {round(train_acc,4)} / {round(test_acc,4)}")


1 / 1.0 / 0.6818
3 / 0.842 / 0.6818
5 / 0.8013 / 0.6753
7 / 0.785 / 0.6753
10 / 0.798 / 0.6688
15 / 0.7915 / 0.6558
20 / 0.7883 / 0.7208
30 / 0.7655 / 0.7208


k / 학습 정확도 / 테스트 정확도

1 / 1.0000 / 0.6818

3 / 0.8420 / 0.6818

5 / 0.8013 / 0.6753

7 / 0.7850 / 0.6753

10 / 0.7980 / 0.6688

15 / 0.7915 / 0.6558

20 / 0.7883 / 0.7208

30 / 0.7655 / 0.7208

분석 질문
1. 학습 정확도가 가장 높은 k값은 무엇인가요?

1입니다. 자기 자신이 가장 가까운 이웃이라 학습데이터를 전부 맞힙니다.

2. 테스트 정확도가 가장 높은 k값은 무엇인가요?

20과 30입니다.(0.7208)

3. k가 작을 때 과대적합이 발생하기 쉬운 이유는 무엇인가요?

k가 작으면 아주 가까운 이웃 몇 개만 보고 예측하므로, 학습 데이터의 국소패턴이나 이상치에 너무 큰 영향을 받습니다. 결정 경계가 복잡해져서 학습데이터는 외우듯 잘 맞히지만, 처음 보는 데이터에서는 성능이 떨어집니다. 학습 1.00, 테스트 0.68이라는 큰 격차가 그 증거입니다.

4. k가 지나치게 커지면 과소적합이 발생할 수 있는 이유는 무엇인가요?

이웃이 너무 많아져 결정 경계가 지나치게 평탄해지고, 가까운 이웃의 정보가 희석되기 때문입니다. 그리고 해당 데이터의 경우 k를 너무 크게 하다보면 결국 데이터가 더 많은쪽(음성)으로 예측이 쏠리게 됩니다. 모델이 지나치게 단순해져서 학습데이터와 테스트 데이터 모두에서 성능이 떨어질 수 있습니다.

5. 학습 정확도만 보고 k값을 선택해도 될까요?

안됩니다. 학습 정확도는 k=1에서 최고지만 테스트 성능은 낮습니다. 학습한 데이터에서만 외우듯 맞히는 것이 중요한게 아니고, 처음보는 데이터에서의 성능도 중요하기 때문에 학습정확도와 테스트 정확도 둘을 함께 봐야 합니다.

## 8. StandardScaler 적용

In [304]:
# StandardScaler 객체 생성
scaler = StandardScaler()

In [305]:
# 학습 데이터로 fit
scaler.fit(X_train_filled)

# 학습 데이터와 테스트 데이터를 각각 transform
X_train_scaled = scaler.transform(X_train_filled)
X_test_scaled = scaler.transform(X_test_filled)

In [306]:
# 변환된 학습 데이터를 DataFrame으로 만들고 describe출력
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
print(X_train_scaled_df.describe())

               preg          plas          pres          skin          insu  \
count  6.140000e+02  6.140000e+02  6.140000e+02  6.140000e+02  6.140000e+02   
mean  -6.943414e-17 -1.099374e-16  3.095606e-16 -3.471707e-17 -4.339634e-18   
std    1.000815e+00  1.000815e+00  1.000815e+00  1.000815e+00  1.000815e+00   
min   -1.153338e+00 -2.190541e+00 -3.924957e+00 -2.480958e+00 -1.559139e+00   
25%   -8.513551e-01 -7.562208e-01 -6.636760e-01 -4.549828e-01 -2.249692e-01   
50%   -2.473897e-01 -1.558076e-01 -1.141979e-02 -4.766137e-03 -1.614373e-01   
75%    6.585583e-01  6.113872e-01  6.408364e-01  3.328964e-01 -9.790537e-02   
max    3.980368e+00  2.579408e+00  4.065181e+00  7.874025e+00  7.703814e+00   

               mass          pedi           age  
count  6.140000e+02  6.140000e+02  6.140000e+02  
mean   1.649061e-16 -1.099374e-16 -1.084908e-16  
std    1.000815e+00  1.000815e+00  1.000815e+00  
min   -2.089621e+00 -1.192094e+00 -1.045895e+00  
25%   -7.073644e-01 -7.042617e-01 -7.9

In [307]:
# 변환된 데이터로 KNN(k=5)학습
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_scaled, y_train)

#학습 정확도와 테스트 정확도 출력
std_train_acc = model.score(X_train_scaled, y_train)
std_test_acc = model.score(X_test_scaled, y_test)
print("학습 정확도:", round(std_train_acc,4))
print("테스트 정확도:", round(std_test_acc,4))

학습 정확도: 0.829
테스트 정확도: 0.7532


In [308]:
# 재현율 출력
pred = model.predict(X_test_scaled)
std_recall = recall_score(y_test, pred)
print("재현율:", round(std_recall,4))

재현율: 0.6111


분석 질문
1. 표준화 후 각 특성의 평균과 표준편차는 대략 얼마인가요?

모든 특성의 평균이 0에 가깝고 표준편차는 1에 가깝습니다.

2. 5번에서 구한 스케일링 전 정확도와 비교하면 어떤 차이가 있나요?

표준화 후 학습 정확도가 0.8013에서 0.8290로 상승했고, 테스트 정확도 또한 0.6753에서 0.7532으로 크게 상승하였습니다. 이는 테스트 데이터에 대한 일반화 성능이 표준화로 인해 개선되었음을 의미합니다.

3. 모델의 알고리즘과 k값은 그대로인데 정확도가 달라지는 이유는 무엇인가요?

KNN은 거리 기반 알고리즘이므로 단위가 큰 특성이 거리 계산을 지배하던 문제가 있었습니다. 표준화를 통해 모든 특성이 동일한 단위(평균 0, 표준편차 1)로 변환되면서 각 특성이 공정하게 거리에 기여하게 되었고, 결과적으로 테스트 정확도가 상승한 것입니다.

## 9. MinMaxScaler 적용 및 비교

In [309]:
# MinMaxScaler로 학습,테스트 데이터 변환
minmax = MinMaxScaler()
minmax.fit(X_train_filled)
X_train_minmax = minmax.transform(X_train_filled)
X_test_minmax = minmax.transform(X_test_filled)


In [310]:
# 변환된 학습 데이터의 describe 출력
X_train_minmax_df = pd.DataFrame(X_train_minmax, columns=X.columns)
X_train_minmax_df.describe()     

,preg,plas,pres,skin,insu,mass,pedi,age
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,0.224660,0.459238,0.491225,0.239591,0.168320,0.291374,0.175246,0.206107
std,0.194950,0.209817,0.125256,0.096651,0.108045,0.139553,0.147127,0.197224
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.058824,0.300699,0.408163,0.195652,0.144033,0.192740,0.071715,0.050000
50%,0.176471,0.426573,0.489796,0.239130,0.150892,0.290389,0.132962,0.133333
75%,0.352941,0.587413,0.571429,0.271739,0.157750,0.374233,0.247327,0.333333
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [311]:
# 변환된 데이터로 KNN(k=5) 학습
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train_minmax, y_train)

# 학습 정확도와 테스트 정확도 출력
mm_train_acc = model.score(X_train_minmax, y_train)
mm_test_acc = model.score(X_test_minmax, y_test)
print("학습 정확도:", round(mm_train_acc,4))
print("테스트 정확도:", round(mm_test_acc,4))

학습 정확도: 0.8322
테스트 정확도: 0.7338


In [312]:
# 재현율 출력
pred = model.predict(X_test_minmax)
mm_recall = recall_score(y_test, pred)
print("재현율:", round(mm_recall,4))

재현율: 0.5556


In [313]:
# 세 가지 전처리 방법의 성능비교
compare_df = pd.DataFrame({
    "KNN 전처리 방법": ["스케일링 없음", "StandardScaler", "MinMaxScaler"],
    "학습 정확도": [base_train_acc, std_train_acc, mm_train_acc],
    "테스트 정확도": [base_test_acc, std_test_acc, mm_test_acc],
    "재현율": [base_recall, std_recall, mm_recall]
})

# 세 가지 방법의 성능 비교
compare_df.round(4)

,KNN 전처리 방법,학습 정확도,테스트 정확도,재현율
0,스케일링 없음,0.8013,0.6753,0.5000
1,StandardScaler,0.8290,0.7532,0.6111
2,MinMaxScaler,0.8322,0.7338,0.5556


분석 질문
1. 테스트 정확도가 가장 높은 전처리 방법은 무엇인가요?

StandardScaler입니다.

2. 재현율이 가장 높은 전처리 방법은 무엇인가요?

StandardScaler입니다.

3. 정확도가 높은 방법과 재현율이 높은 방법이 다를 수도 있나요?

네 다를 수 있습니다. 정확도는 전체 데이터를 대상으로 계산하지만, 재현율은 실제 양성 데이터만 대상으로 계산합니다. 따라서 다수인 음성데이터만 많이 맞혀도 정확도는 높아질 수 있지만, 실제 양성을 많이 놓치면 재현율은 낮아지게 됩니다.

4. 스케일링을 적용한다고 해서 정확도가 반드시 높아지는 것은 아닌 이유는 무엇일까요?

스케일링은 각 특성이 거리 계산에 미치는 영향을 조정할 뿐이며, 새로운 정보를 추가하지는 않기 때문입니다. 숫자가 큰 특성이 원래 중요한 정보를 담고 있었다면 오히려 영향력이 줄어들 수도 있고, 테스트 데이터가 154개로 적어서 몇 개만 달라져도 정확도가 우연히 오르내릴 수 있습니다. 또 MinMaxScaler는 이상치가 있으면 나머지 값이 좁은 구간에 몰리는 단점이 있습니다.

## 10. 종합 - KNN 설정 선택 및 최종 분석

스케일링 방법 선택

StandardScaler

In [314]:
# k값 실험
final_results = []
scaler = StandardScaler().fit(X_train_filled)

# 학습·테스트 데이터를 같은 기준으로 변환
X_tr_scaled = scaler.transform(X_train_filled)
X_te_scaled = scaler.transform(X_test_filled)

# k값별 학습 정확도·테스트 정확도·재현율 기록
ks = [1, 3, 5, 7, 10, 15, 20, 30]

for k in ks:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_tr_scaled, y_train)
    pred = model.predict(X_te_scaled)

    train_acc = model.score(X_tr_scaled, y_train)
    test_acc = model.score(X_te_scaled, y_test)
    rec = recall_score(y_test, pred)

    final_results.append({
        "k": k,
        "학습 정확도": round(train_acc,4),
        "테스트 정확도": round(test_acc,4),
        "재현율": round(rec,4)
    })

# 결과를 표로 정리
result_df = pd.DataFrame(final_results)
result_df

,k,학습 정확도,테스트 정확도,재현율
0,1,1.0000,0.7208,0.5556
1,3,0.8469,0.7208,0.6111
2,5,0.8290,0.7532,0.6111
3,7,0.8176,0.7273,0.5556
4,10,0.8046,0.7403,0.5370
5,15,0.8029,0.7403,0.5556
6,20,0.7883,0.7597,0.5556
7,30,0.7883,0.7403,0.5185


최종 설정 선택

1. 사용할 스케일링 방법 : StandardScaler

특성마다 단위,범위가 달라(인슐린 0~846, 나이 21~81) 거리 기반 KNN에는 스케일링이 필수였고, MinMaxScaler보다 StandardScaler가 이상치에 덜 흔들려 StandardScaler를 선택하였습니다.


2. 사용할 k 값 : 5

질병과 관련된 경우에는 앞선 문제에서 답한 이유로 재현율이 중요한 분야이기 때문에 재현율이 가장 높은 3,5를 선택하였고, 재현율이 일치하는 둘 중에서는 테스트 정확도가 더 높은 5를 최종선택하였습니다.


In [315]:
# 최종 설정으로 KNN 학습 및 평가
final_model = KNeighborsClassifier(n_neighbors=5)
final_model.fit(X_tr_scaled, y_train)

print("학습 정확도:", round(final_model.score(X_tr_scaled, y_train), 4))
print("테스트 정확도:", round(final_model.score(X_te_scaled, y_test), 4))

y_pred_final = final_model.predict(X_te_scaled)

print(confusion_matrix(y_test, y_pred_final))
print("정밀도:", round(precision_score(y_test, y_pred_final), 4))
print("재현율:", round(recall_score(y_test, y_pred_final), 4))
print("F1:", round(f1_score(y_test, y_pred_final), 4))


학습 정확도: 0.829
테스트 정확도: 0.7532
[[83 17]
 [21 33]]
정밀도: 0.66
재현율: 0.6111
F1: 0.6346


최종 분석 보고서

1. 최종적으로 어떤 스케일링 방법과 k값을 선택했나요?

StandardScaler, k=5를 선택했습니다.

2. 해당 설정을 선택한 이유는 무엇인가요?

KNN은 거리로 이웃을 찾기 때문에 범위가 큰 특성이 지배적인 영향력을 가집니다. insu(0~846)같이 범위가 큰 특성이 지배적으로 반영되고, age(21~81)처럼 범위가 작은 특성은 무시될 수도 있습니다. 그래서 위 데이터의 분석에는 스케일링이 필요합니다.
한편, MinMaxScaler의 경우엔 최솟값·최댓값에 결과가 묶여 insu의 큰 값 하나에 전체가 눌리는 문제를 또 보이고 있습니다.
따라서 특성들을 평균 0, 표준편차 1로 맞추는 StandardScaler를 이용하여 이상치에 덜 왜곡되는 방법을 선택하였습니다.

그리고 k를 고를때는 재현율을 우선 봐야한다고 생각했습니다. 병과 관련된 데이터이기 때문에 실제 아픈사람이 진단을 올바르게 받을 확률을 올리는 것이 무엇보다 중요하다고 생각했습니다. 다시 말해 당뇨병 양성임에도 치료 기회를 잃게 되는 경우를 줄이는 것이(놓친 양성 FN을 줄이는 것이) 핵심이라고 생각했습니다. 이런 이유에서 재현율이 가장 높으면서도 그중에서 테스트정확도가 높은 k=5를 선택하였습니다.
또 k=5의 경우 테스트 정확도가 가장 높은 k=20(0.7597)과도 테스트 정확도가 크게 차이나지 않습니다.(0.7532)

3. 스케일링 전후 KNN의 성능은 어떻게 달라졌나요?

스케일링 전(문제 5·7 결과)과 비교하면 전반적으로 성능이 개선되었습니다. 같은 k=5에서 테스트 정확도가 0.6753에서 0.7532로 상승하였고, 재현율 또한 0.5000에서 0.6111로 상승하였습니다. 낮은 k 구간에서 성능 변동도 줄었습니다.

4. k값이 달라지면서 정확도는 어떤 변화를 보였나요?

k=1에서 학습 정확도는 100프로(자기 자신이 이웃)인데 테스트 정확도는 차이가 너무 커서 전형적인 과대적합 경향을 보였습니다. k가 커질수록 두 정확도의 차이는 줄어들지만, 너무 커지면 음성 쪽으로 예측이 쏠려 성능이 떨어지는 과소적합 경향이 나타나기도 했습니다.

5. 정확도와 재현율 중 어떤 지표를 더 중요하게 고려했나요? 이 이유를 설명하세요.

재현율을 더 중요하게 봤습니다. 당뇨 검사에서는 양성(환자)을 음성으로 놓치는 FN이 가장 위험하므로 재현율을 우선해야 합니다. 그리고 테스트 데이터가 음성 100명 : 양성 54명으로 기울어 있어(전체 양성 비율 약 34.9%) 음성만 모두 맞혀도 정확도가 65%를 넘습니다. 즉 정확도는 다수 클래스에 가려지기 쉽습니다. 

6. 결측치 처리와 스케일링에서 학습 데이터에만 기준을 맞춰야 하는 이유는 무엇인가요?

테스트 데이터는 처음 보는 데이터라고 가정해야 공정한 평가가 되기 때문입니다. 테스트 데이터의 중앙값이나 평균 등을 사용하게 되면 테스트 정보가 새어 들어가는 데이터 누수가 생겨 성능이 실제보다 좋게 나오게 됩니다. 그러면 성능이 낙관적으로 나오게 되고 올바른 정보가 되어주지 못합니다. 그리고 해당 문제의 경우를 예로 들어 실제 상황을 말해보자면, 새 환자가 들어올 경우 당연히 미리 알 수 없었던 전혀 새로운 정보를 받게 될 것이기 때문입니다.

7. 이번 과제를 진행하면서 머신러닝 전체 과정 중 가장 중요하다고 생각한 단계는 무엇인가요?

데이터를 학습 데이터와 테스트 데이터로 나누고, 그 뒤에 학습 데이터를 기준으로 전처리하는 '데이터 분할 과정'이 무엇보다도 중요하다고 생각했습니다. 일단 우리가 인공지능에 대해 기대하는 것이 단순히 암기를 잘하는 데이터 저장통이 아니기 때문입니다. 단순히 암기된 데이터가 아닌 새로운 데이터에서의 성능이 중요하고, 그 성능을 올바르게 평가하기 위해선 데이터 분할단계와 데이터누수를 막는 전처리 방법들이 중요하다고 생각했습니다. 그리고 이 단계에서 잘못된다면 뒤에 어떤 조정처리를 하든 이미 왜곡된 데이터와 준비물들을 가지고 조정하는 것이라, 어차피 이후 받아보는 데이터들도 다 신뢰할 수 없을 뿐이라고 생각했습니다.